# 🎬 Kaggle Notebook 4 — Final Demo
## Quantum-Enhanced LPR — Standalone Presentation Notebook

**Self-contained demo** — pulls everything from HuggingFace Hub, shows side-by-side Quantum vs Classical predictions.

**Secrets required:** `HF_TOKEN_1`, `HF_TOKEN_2`

In [ ]:
!pip install pennylane pennylane-lightning huggingface_hub editdistance -q

import os, json, zipfile, time
import torch, torch.nn as nn
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import pennylane as qml
import editdistance
from huggingface_hub import HfApi, login, whoami, hf_hub_download

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}  |  PennyLane: {qml.__version__}')

In [ ]:
# ── HF Auth ──────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    s=UserSecretsClient()
    HF_TOKENS=[s.get_secret('HF_TOKEN_1'),s.get_secret('HF_TOKEN_2')]
except Exception:
    HF_TOKENS=['ADD_YOUR_HF_TOKEN_1_IN_KAGGLE_SECRETS','ADD_YOUR_HF_TOKEN_2_IN_KAGGLE_SECRETS']

HF_USERNAME='Shanmuk4622'
HF_DATASET_REPO=f'{HF_USERNAME}/quantum-lpr-dataset'
HF_MODEL_REPO=f'{HF_USERNAME}/quantum-lpr-checkpoints'
active_token=None; api=None
for tok in HF_TOKENS:
    try:
        login(token=tok,add_to_git_credential=False)
        info=whoami(token=tok); active_token=tok; api=HfApi(token=tok)
        print(f'✅ HF: {info["name"]}'); break
    except Exception: pass
if not active_token: raise RuntimeError('HF auth failed')

WORK_DIR='/kaggle/working'; DATA_DIR=f'{WORK_DIR}/lpr_data'; CKPT_DIR=f'{WORK_DIR}/ckpt'
os.makedirs(DATA_DIR,exist_ok=True); os.makedirs(CKPT_DIR,exist_ok=True)
ZIP_L=f'{DATA_DIR}/train.zip'; CSV_L=f'{DATA_DIR}/labels.csv'; EXT_DIR=f'{DATA_DIR}/images'
CHARS='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CHAR2IDX={c:i+1 for i,c in enumerate(CHARS)}; IDX2CHAR={i+1:c for i,c in enumerate(CHARS)}
N_QUBITS=8; N_LAYERS=2; SEED=42

for hfn,loc in [('data/2_train_hr_images.csv',CSV_L),('data/wYe7pBJ7-train.zip',ZIP_L)]:
    if not os.path.exists(loc):
        print(f'⬇️  {hfn}...')
        hf_hub_download(repo_id=HF_DATASET_REPO,filename=hfn,repo_type='dataset',
                        token=active_token,local_dir=DATA_DIR,local_dir_use_symlinks=False)
if not os.path.exists(EXT_DIR):
    print('📂 Extracting...'); 
    with zipfile.ZipFile(ZIP_L) as z: z.extractall(EXT_DIR)
print('✅ Data ready')

In [ ]:
# ── Models ──────────────────────────────────────────────
class ZeroDCE_Light(nn.Module):
    def __init__(self): super().__init__(); self.conv=nn.Sequential(nn.Conv2d(3,16,3,1,1),nn.ReLU(),nn.Conv2d(16,16,3,1,1),nn.ReLU(),nn.Conv2d(16,24,3,1,1),nn.Tanh())
    def forward(self,x):
        A,e=self.conv(x),x
        for i in range(8): e=e+A[:,3*i:3*(i+1)]*(torch.pow(e,2)-e)
        return e

d_qml=qml.device('default.qubit',wires=N_QUBITS)
@qml.qnode(d_qml,interface='torch')
def qc(inputs,weights):
    qml.templates.AngleEmbedding(inputs,wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights,wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QL(nn.Module):
    def __init__(self): super().__init__(); self.q=qml.qnn.TorchLayer(qc,{'weights':(N_LAYERS,N_QUBITS,3)})
    def forward(self,x): return self.q(x)

class HybridLPRNet_8Q(nn.Module):
    def __init__(self,nc=37):
        super().__init__()
        self.enhancer=ZeroDCE_Light()
        self.features=nn.Sequential(nn.Conv2d(3,64,3,1,1),nn.MaxPool2d(2),nn.ReLU(),nn.Conv2d(64,128,3,1,1),nn.MaxPool2d(2),nn.ReLU(),nn.Conv2d(128,N_QUBITS,1,1))
        self.quantum=QL(); self.rnn=nn.LSTM(N_QUBITS,128,bidirectional=True,batch_first=True); self.cls=nn.Linear(256,nc)
    def forward(self,x):
        x=self.enhancer(x); x=self.features(x); b,c,h,w=x.size()
        xs=x.mean(dim=2).permute(0,2,1); q=self.quantum(xs.reshape(-1,N_QUBITS)).reshape(b,w,N_QUBITS)
        return self.cls(self.rnn(q)[0]).permute(1,0,2)

class ClassicalLPRNet(nn.Module):
    def __init__(self,nc=37):
        super().__init__()
        self.enhancer=ZeroDCE_Light()
        self.features=nn.Sequential(nn.Conv2d(3,64,3,1,1),nn.MaxPool2d(2),nn.ReLU(),nn.Conv2d(64,128,3,1,1),nn.MaxPool2d(2),nn.ReLU(),nn.Conv2d(128,N_QUBITS,1,1))
        self.cl=nn.Sequential(nn.Linear(N_QUBITS,16),nn.Tanh(),nn.Linear(16,N_QUBITS))
        self.rnn=nn.LSTM(N_QUBITS,128,bidirectional=True,batch_first=True); self.cls=nn.Linear(256,nc)
    def forward(self,x):
        x=self.enhancer(x); x=self.features(x); b,c,h,w=x.size()
        return self.cls(self.rnn(self.cl(x.mean(dim=2).permute(0,2,1)))[0]).permute(1,0,2)

q_model=HybridLPRNet_8Q(37).to(device); c_model=ClassicalLPRNet(37).to(device)
for mdl,best,latest,name in [(q_model,'quantum/best.pth','quantum/latest.pth','Quantum ⚡'),
                               (c_model,'classical/best.pth','classical/latest.pth','Classical 🔷')]:
    for hfp in [best,latest]:
        try:
            f=hf_hub_download(repo_id=HF_MODEL_REPO,filename=hfp,repo_type='model',
                              token=active_token,local_dir=CKPT_DIR,local_dir_use_symlinks=False,force_download=True)
            ck=torch.load(f,map_location=device); mdl.load_state_dict(ck['model_state_dict'])
            print(f'  ✅ {name}: {hfp}  (ep {ck.get("epoch","?")}, CER={ck.get("val_cer","?")})')
            break
        except Exception: pass
    mdl.eval()
print('✅ Models ready')

In [ ]:
# ── Dataset + Decoder ─────────────────────────────────────
class LPRDataset(Dataset):
    def __init__(self,csv,root,tf,night=False):
        self.df=pd.read_csv(csv); self.root=root; self.tf=tf; self.night=night
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; p=str(r['path'])
        f=p if p.startswith('/') else os.path.join(self.root,p)
        if not os.path.exists(f): f=os.path.join(EXT_DIR,'train',os.path.basename(p))
        try: img=Image.open(f).convert('RGB')
        except: img=Image.new('RGB',(256,64))
        if self.tf: img=self.tf(img)
        if self.night:
            g=np.random.uniform(2.0,3.5)
            img=torch.clamp(torch.pow(img,g)+torch.randn_like(img)*0.05,0,1)
        lbl=str(r['label']); enc=[CHAR2IDX[c] for c in lbl.upper() if c in CHAR2IDX]
        return img,lbl

tf=transforms.Compose([transforms.Resize((64,256)),transforms.ToTensor()])
torch.manual_seed(SEED)
tmp=LPRDataset(CSV_L,EXT_DIR,tf); N=len(tmp); nt=int(N*0.70); nv=int(N*0.15)
_,_,test_sub=random_split(range(N),[nt,nv,N-nt-nv])
TEST_IDX=list(test_sub)

def ctc_dec(lp):
    idx=torch.argmax(lp,dim=2); out=[]
    for b in range(idx.size(1)):
        s,chars,prev=idx[:,b].tolist(),[],- 1
        for i in s:
            if i!=0 and i!=prev: chars.append(IDX2CHAR.get(i,''))
            prev=i
        out.append(''.join(chars))
    return out

print(f'✅ Test set: {N-nt-nv} samples')

In [ ]:
# ── 🌙 Night-Time Recognition Demo ─────────────────────
N_DEMO=12; np.random.seed(SEED)
demo_idx=np.random.choice(TEST_IDX,N_DEMO,replace=False)
clean_ds=LPRDataset(CSV_L,EXT_DIR,tf,night=False)
night_ds=LPRDataset(CSV_L,EXT_DIR,tf,night=True)

fig=plt.figure(figsize=(24,3.8*N_DEMO)); fig.patch.set_facecolor('#0D0D0D')
outer=fig.add_gridspec(N_DEMO+1,5,hspace=0.05,wspace=0.04,top=0.97,bottom=0.02,left=0.005,right=0.995)
HDRS=['Clean Original','Night Input 🌙','ZeroDCE Enhanced','⚡ Quantum Result','🔷 Classical Result']
HCOLS=['#455A64','#1A237E','#004D40','#4A148C','#0D47A1']
for j,(h,hc) in enumerate(zip(HDRS,HCOLS)):
    a=fig.add_subplot(outer[0,j]); a.set_facecolor(hc); a.axis('off')
    a.text(0.5,0.5,h,ha='center',va='center',color='white',fontsize=11,fontweight='bold')

qc_ok=cc_ok=0
for row,idx in enumerate(demo_idx):
    cimg,lbl=clean_ds[int(idx)]; nimg,_=night_ds[int(idx)]
    inp=nimg.unsqueeze(0).to(device)
    with torch.no_grad():
        enh=q_model.enhancer(inp).cpu().squeeze(0)
        qp=ctc_dec(q_model(inp).cpu())[0]
        cp=ctc_dec(c_model(inp).cpu())[0]
    qok=(qp==lbl); cok=(cp==lbl)
    if qok: qc_ok+=1
    if cok: cc_ok+=1
    def sa(c): a=fig.add_subplot(outer[row+1,c]); a.axis('off'); return a
    def si(a,t,ttl='',tclr='#aaa'):
        a.imshow(np.clip(t.permute(1,2,0).numpy(),0,1))
        a.set_title(ttl,color=tclr,fontsize=8,pad=2)
        [s.set_edgecolor('#333') for s in a.spines.values()]
        a.tick_params(left=False,bottom=False,labelleft=False,labelbottom=False)
    si(sa(0),cimg,f'True: {lbl}')
    si(sa(1),nimg,'Night Dark','#7986CB')
    si(sa(2),enh,'ZeroDCE Enhanced','#80CBC4')
    for col,pred,ok in [(3,qp,qok),(4,cp,cok)]:
        a=sa(col); a.set_facecolor('#1B5E20' if ok else '#7F0000')
        a.text(0.5,0.5,f"{'✅' if ok else '❌'}  {pred or '(blank)'}\nTrue: {lbl}",
               ha='center',va='center',color='white',fontsize=12,fontweight='bold')

plt.suptitle('🌙 Night-Time License Plate Recognition — Quantum ⚡ vs Classical 🔷',
             fontsize=15,fontweight='bold',y=1.005,color='white')
plt.tight_layout()
demo_path='/kaggle/working/demo_result.png'
plt.savefig(demo_path,dpi=120,bbox_inches='tight',facecolor='#0D0D0D'); plt.show()
try:
    api.upload_file(path_or_fileobj=demo_path,path_in_repo='results/demo_result.png',
                    repo_id=HF_MODEL_REPO,repo_type='model',token=active_token)
except Exception: pass

print(f'\n✅ Demo complete!')
print(f'   ⚡ Quantum  correct: {qc_ok}/{N_DEMO}  ({qc_ok/N_DEMO*100:.0f}%)')
print(f'   🔷 Classical correct: {cc_ok}/{N_DEMO}  ({cc_ok/N_DEMO*100:.0f}%)')

# Load full comparison table if available
try:
    tf_p=hf_hub_download(repo_id=HF_MODEL_REPO,filename='results/final_comparison_table.csv',
                          repo_type='model',token=active_token,local_dir='/kaggle/working',
                          local_dir_use_symlinks=False,force_download=True)
    tbl=pd.read_csv(tf_p)
    print('\n'+'-'*55+'\n  FULL COMPARISON TABLE (from Notebook 2)\n'+'-'*55)
    print(tbl.to_string(index=False))
except Exception:
    print('\n(Run 02_Evaluation_Suite_Kaggle.ipynb for full metrics table)')

---
## 📖 Architecture Summary
```
Night Image → ZeroDCE Enhancer → CNN (→8ch) → 8-Qubit Circuit → Bi-LSTM → CTC Decode → Text
                                                ↕
                               AngleEmbedding + StronglyEntanglingLayers
                               256-dim Hilbert space (2^8) vs classical 8-dim
```
All checkpoints, metrics, and figures are stored at:
- **HF Model Repo:** `Shanmuk4622/quantum-lpr-checkpoints`
- **HF Dataset Repo:** `Shanmuk4622/quantum-lpr-dataset`